<a href="https://colab.research.google.com/github/dinujakr/Statistical-Learning-e20190/blob/main/Assignments/GPR_LR_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.

In [1]:
import kagglehub

# Download latest version
kagglepath="elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'eergy-efficiency-dataset' dataset.
Path to dataset files: /kaggle/input/eergy-efficiency-dataset


In [2]:
import os
import pandas as pd
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/ENB2012_data.csv")

Listing contents of: /kaggle/input/eergy-efficiency-dataset
ENB2012_data.csv


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load the data (using the df2 dataframe you already read)
# Extract features (X1 to X8) and targets (Y1, Y2)
X_orig = df2.iloc[:, :8].values
y1 = df2.iloc[:, 8].values  # Heating Load
y2 = df2.iloc[:, 9].values  # Cooling Load

# 2. Stack the datasets to create a single-parameter GP target
# Create task indicators: 0 for Heating Load, 1 for Cooling Load
X_y1_task = np.hstack((X_orig, np.zeros((X_orig.shape[0], 1))))
X_y2_task = np.hstack((X_orig, np.ones((X_orig.shape[0], 1))))

# Combine features and targets
X_stacked = np.vstack((X_y1_task, X_y2_task))
y_stacked = np.concatenate((y1, y2))

# 3. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_stacked, y_stacked, test_size=0.2, random_state=42)

# 4. Scale the features (important for GP distance metrics)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Stacked Training shape: {X_train_scaled.shape}")
print(f"Stacked Testing shape: {y_test.shape}")

Stacked Training shape: (1228, 9)
Stacked Testing shape: (308,)


In [4]:
# Define a composite kernel: Constant * RBF
# We allow length scales to adapt to the features
kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(X_train_scaled.shape[1]), length_scale_bounds=(1e-2, 1e2))

# Initialize GPR with noise handling (alpha)
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, alpha=0.1, random_state=42)

# Fit the model to the stacked single-parameter dataset
print("Fitting Gaussian Process Regression Model... (This may take a moment)")
gpr.fit(X_train_scaled, y_train)
print("Model optimized successfully!")
print(f"Learned Kernel: {gpr.kernel_}")

Fitting Gaussian Process Regression Model... (This may take a moment)


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 3 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 4 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Model optimized successfully!
Learned Kernel: 12.2**2 * RBF(length_scale=[0.695, 100, 1.13, 100, 100, 1.57, 10, 0.9, 2.72])


In [5]:
# Make predictions
y_pred, sigma = gpr.predict(X_test_scaled, return_std=True)

# Separate results back into Heating and Cooling for detailed analysis
test_task_indicators = X_test[:, -1]

# Heating Load Evaluation (Indicator = 0)
heating_mask = (test_task_indicators == 0)
mse_h = mean_squared_error(y_test[heating_mask], y_pred[heating_mask])
r2_h = r2_score(y_test[heating_mask], y_pred[heating_mask])

# Cooling Load Evaluation (Indicator = 1)
cooling_mask = (test_task_indicators == 1)
mse_c = mean_squared_error(y_test[cooling_mask], y_pred[cooling_mask])
r2_c = r2_score(y_test[cooling_mask], y_pred[cooling_mask])

# Overall Evaluation
print("\n=== Performance Metrics ===")
print(f"Overall R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"Overall MSE:      {mean_squared_error(y_test, y_pred):.4f}\n")
print(f"Heating Load ($Y_1$) - MSE: {mse_h:.4f}, R²: {r2_h:.4f}")
print(f"Cooling Load ($Y_2$) - MSE: {mse_c:.4f}, R²: {r2_c:.4f}")


=== Performance Metrics ===
Overall R² Score: 0.9959
Overall MSE:      0.3809

Heating Load ($Y_1$) - MSE: 0.2126, R²: 0.9979
Cooling Load ($Y_2$) - MSE: 0.5470, R²: 0.9935


# Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand'  using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

In [6]:
import kagglehub

# Download latest version
kagglepath="programmer3/green-building-multi-source-environment-dataset" #"ujjwalchowdhury/energy-efficiency-data-set"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'green-building-multi-source-environment-dataset' dataset.
Path to dataset files: /kaggle/input/green-building-multi-source-environment-dataset


In [7]:
import os
import pandas as pd

print(f"Listing contents of: {path}")
!ls {path}

# Read the dataset safely without the inspector error
df2 = pd.read_csv(path + "/green_building_dataset.csv")

Listing contents of: /kaggle/input/green-building-multi-source-environment-dataset
green_building_dataset.csv


In [8]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 1. Load the dataset (using df2 as already loaded in your notebook)
# Drop any completely non-numeric or identifier columns if they exist (e.g., IDs, timestamps)
numeric_df = df2.select_dtypes(include=[np.number])

# 2. Compute correlations with the target variable
target_col = 'predicted_energy_demand'
correlations = numeric_df.corr()[target_col].sort_values(ascending=False)

print("=== Correlation with Predicted Energy Demand ===")
print(correlations)

# 3. Automatically select features with a correlation absolute value > 0.1 (excluding the target itself)
threshold = 0.1
selected_features = correlations[abs(correlations) > threshold].index.drop(target_col).tolist()

print(f"\nSelected Features for Linear Regression (Correlation > {threshold}):")
print(selected_features)

=== Correlation with Predicted Energy Demand ===
predicted_energy_demand    1.000000
ventilation_rate           0.728865
electricity_consumption    0.398703
cooling_energy             0.370632
heating_energy             0.271304
equipment_load             0.058766
occupancy                  0.057655
activity_level             0.018522
wind_speed                 0.011333
indoor_humidity            0.007899
outdoor_temperature        0.006786
outdoor_humidity           0.006451
solar_radiation            0.005331
predicted_comfort_index    0.003568
rainfall                  -0.004161
indoor_temperature        -0.008106
indoor_lighting           -0.020631
indoor_noise              -0.024454
co2_concentration         -0.036466
Name: predicted_energy_demand, dtype: float64

Selected Features for Linear Regression (Correlation > 0.1):
['ventilation_rate', 'electricity_consumption', 'cooling_energy', 'heating_energy']


In [9]:
# 1. Define X and y based on selected features
X = df2[selected_features]
y = df2[target_col]

# Handle any missing values if present
X = X.fillna(X.mean())
y = y.fillna(y.mean())

# 2. Split into Train and Test sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Initialize and fit the Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# 5. Predict on test set
y_pred = lr_model.predict(X_test_scaled)

# 6. Evaluate Model Performance
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=== Linear Regression Performance ===")
print(f"R² Score (Variance Explained): {r2:.4f}")
print(f"Mean Absolute Error (MAE):    {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

# 7. Print Coefficients to interpret feature impact
print("\n=== Model Coefficients (Feature Importance) ===")
for feature, coef in zip(selected_features, lr_model.coef_):
    print(f"{feature:<30} : {coef:.4f}")
print(f"{'Intercept':<30} : {lr_model.intercept_:.4f}")

=== Linear Regression Performance ===
R² Score (Variance Explained): 0.9491
Mean Absolute Error (MAE):    1.7166
Root Mean Squared Error (RMSE): 2.1806

=== Model Coefficients (Feature Importance) ===
ventilation_rate               : 7.1413
electricity_consumption        : 4.1716
cooling_energy                 : 3.5548
heating_energy                 : 2.8328
Intercept                      : 33.7360
